# Inference: ймовірності моделей для val і test

Етап 8. Тут лише **рахуємо** ймовірності двох навчених моделей — усі рішення (пороги, сабміт) приймаються
локально в `08_pipeline.ipynb`, без GPU і без повторних запусків на Kaggle.

| що | для чого | файл у Output |
|---|---|---|
| класифікатор на test | поріг класифікатора на test | `test_classifier_probs.csv` |
| U-Net на **всій** val (19 256, і порожні) | підбір порогів пайплайну | `val_unet_probs.npz` |
| U-Net на всьому test (15 606) | сабміт | `test_unet_probs.npz` |

Ймовірності класифікатора для val уже є (`data/classifier_v1/val_probs.csv`). Ймовірності U-Net — `uint8`
(p × 255), як у `06`.

Локально ноутбук проходить на `data/train_micro/` (перевірка логіки): «test» там — кілька знімків train_micro.

**На Kaggle:**
1. Settings → Accelerator → **GPU**, Internet → **On**; Add-ons → Secrets → `KAGGLE_API_TOKEN` **attached**.
2. Input: `airbus-ship-detection-src`, виходи ноутбуків **`airbus-unet`** (ваги U-Net) і
   **`airbus-ship-detection-outside-competition`** (ваги класифікатора). Змагання в Input **не** додавати.
3. **Save Version → Save & Run All**.

## 0. Дані на Kaggle

Як у `05`/`06`, але розпаковуємо ще й `test_v2/`.

In [ ]:
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
KAGGLE_DATA_DIR = Path("/kaggle/tmp/data")

if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["KAGGLE_API_TOKEN"] = UserSecretsClient().get_secret("KAGGLE_API_TOKEN")
    # Встановлена в Kaggle версія kaggle може не знати нового формату токена.
    !pip install -q kaggle==2.2.4 segmentation-models-pytorch==0.5.0

In [ ]:
if ON_KAGGLE and not (KAGGLE_DATA_DIR / "test_v2").exists():
    !mkdir -p /kaggle/tmp && cd /kaggle/tmp && kaggle competitions download -c airbus-ship-detection
    !unzip -q /kaggle/tmp/airbus-ship-detection.zip "train_v2/*" "test_v2/*" train_ship_segmentations_v2.csv sample_submission_v2.csv -d {KAGGLE_DATA_DIR}

if ON_KAGGLE:
    n_train = sum(1 for _ in (KAGGLE_DATA_DIR / "train_v2").iterdir())
    n_test = sum(1 for _ in (KAGGLE_DATA_DIR / "test_v2").iterdir())
    print("знімків у train_v2:", n_train, "| у test_v2:", n_test)
    assert n_train == 192556 and n_test == 15606

## 1. Налаштування і знімки

Val — та сама `make_split` (seed 42), але тепер **усі** val-знімки, і порожні теж: на них перевіряється,
чи U-Net не «малює» кораблі там, куди її пропустив класифікатор. Список test-знімків — з `sample_submission_v2.csv`.
Шляхи до ваг шукаємо в `/kaggle/input` за назвою файлу.

In [ ]:
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torchvision
from torch.utils.data import DataLoader

if ON_KAGGLE:
    sys.path.append("/kaggle/input/datasets/yurapolishchuk/airbus-ship-detection-src")
    DATA_DIR = KAGGLE_DATA_DIR
    VAL_DIR = DATA_DIR / "train_v2"
    TEST_DIR = DATA_DIR / "test_v2"
    OUT_DIR = Path("/kaggle/working")
    CLASSIFIER_WEIGHTS = next(Path("/kaggle/input").rglob("classifier_resnet34.pth"))
    UNET_WEIGHTS = next(Path("/kaggle/input").rglob("unet_resnet34.pth"))
    DEVICE = "cuda"
    NUM_WORKERS = 4
else:
    ROOT = Path.cwd().parent
    sys.path.append(str(ROOT / "src"))
    DATA_DIR = ROOT / "data"
    VAL_DIR = DATA_DIR / "train_micro"
    TEST_DIR = DATA_DIR / "train_micro"
    OUT_DIR = DATA_DIR / "inference_local"
    CLASSIFIER_WEIGHTS = DATA_DIR / "classifier_v1" / "classifier_resnet34.pth"
    UNET_WEIGHTS = DATA_DIR / "unet_v1" / "unet_resnet34.pth"
    DEVICE = "cpu"
    NUM_WORKERS = 0

from dataset import ShipClassificationDataset, ShipSegmentationDataset
from split import make_split

SEED = 42
BATCH_SIZE = 32
USE_AMP = DEVICE == "cuda"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Kaggle:", ON_KAGGLE, "| device:", DEVICE)
print("класифікатор:", CLASSIFIER_WEIGHTS)
print("U-Net:", UNET_WEIGHTS)

In [ ]:
masks = pd.read_csv(DATA_DIR / "train_ship_segmentations_v2.csv")
if ON_KAGGLE:
    split = make_split(masks)
    test_ids = pd.read_csv(DATA_DIR / "sample_submission_v2.csv")["ImageId"]
else:
    available = sorted(p.name for p in VAL_DIR.iterdir())
    split = make_split(masks[masks["ImageId"].isin(available)], val_size=0.2)
    test_ids = pd.Series(available[:8])

val_df = split[split["split"] == "val"].reset_index(drop=True)
print(f"val: {len(val_df)} знімків ({(val_df['n_ships'] > 0).mean():.1%} з кораблями) | test: {len(test_ids)}")

## 2. Класифікатор на test

Архітектура — як у `03_classifier_train.ipynb`: ResNet34, останній шар → 1 логіт.
Спершу перевірка ваг: на перших 512 val-знімках p має збігтися з `val_probs.csv` з навчання
(лише на Kaggle — локальний поділ інший). Невелика різниця (~1e-3) — нормально: AMP рахує в
половинній точності, і результат трохи залежить від складу батча.

In [ ]:
classifier = torchvision.models.resnet34()
classifier.fc = torch.nn.Linear(classifier.fc.in_features, 1)
classifier.load_state_dict(torch.load(CLASSIFIER_WEIGHTS, map_location=DEVICE))
classifier = classifier.to(DEVICE).eval()


@torch.no_grad()
def predict_classifier(image_ids, image_dir):
    dataset = ShipClassificationDataset(image_ids, np.zeros(len(image_ids)), image_dir, train=False)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    probs = []
    for images, _ in loader:
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = classifier(images.to(DEVICE)).squeeze(1)
        probs.append(torch.sigmoid(logits.float()).cpu())
    return torch.cat(probs).numpy()


if ON_KAGGLE:
    saved = next(Path("/kaggle/input").rglob("val_probs.csv"))
    saved = pd.read_csv(saved).head(512)
    check = predict_classifier(saved["ImageId"], VAL_DIR)
    print("макс. різниця p з val_probs.csv на 512 знімках:", np.abs(check - saved["p"].values).max())

start = time.time()
test_classifier = pd.DataFrame({"ImageId": test_ids, "p": predict_classifier(test_ids, TEST_DIR)})
test_classifier.to_csv(OUT_DIR / "test_classifier_probs.csv", index=False)
print(f"test: {len(test_classifier)} знімків, {time.time() - start:.0f} с")
print(f"частка test з p > 0.5: {(test_classifier['p'] > 0.5).mean():.1%}")

## 3. U-Net на val і test

`ShipSegmentationDataset` повертає ще й маску — тут вона не потрібна (для test масок і немає: порожній словник
RLE дає нульові маски). Ймовірності зберігаються стиснутими: на порожніх знімках майже всі пікселі = 0.

In [ ]:
unet = smp.Unet(encoder_name="resnet34", encoder_weights=None, classes=1)
unet.load_state_dict(torch.load(UNET_WEIGHTS, map_location=DEVICE))
unet = unet.to(DEVICE).eval()


@torch.no_grad()
def predict_unet(image_ids, image_dir):
    dataset = ShipSegmentationDataset(image_ids, {}, image_dir, train=False)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    batches = []
    for images, _ in loader:
        with torch.autocast(device_type=DEVICE, enabled=USE_AMP):
            logits = unet(images.to(DEVICE))
        probs = torch.sigmoid(logits.float())[:, 0]
        batches.append((probs * 255).round().to(torch.uint8).cpu().numpy())
    return np.concatenate(batches)


for name, image_ids, image_dir in [("val", val_df["ImageId"], VAL_DIR), ("test", test_ids, TEST_DIR)]:
    start = time.time()
    probs = predict_unet(image_ids, image_dir)
    np.savez_compressed(OUT_DIR / f"{name}_unet_probs.npz", probs=probs, image_ids=np.asarray(image_ids))
    has_pixels = (probs > 0.7 * 255).any(axis=(1, 2)).mean()
    print(f"{name}: {probs.shape}, {time.time() - start:.0f} с | знімків з хоч одним пікселем p > 0.7: {has_pixels:.1%}")

for path in sorted(OUT_DIR.glob("*_unet_probs.npz")):
    print(f"{path.name}: {path.stat().st_size / 1e6:.0f} МБ")

### 3.1. Перевірка очима на test

4 test-знімки з найбільшим p класифікатора: знімок, p і маска U-Net (p > 0.7, зелене). Розмітки для test
немає — дивимось лише, чи маска лежить на чомусь схожому на корабель.

In [ ]:
test_probs = np.load(OUT_DIR / "test_unet_probs.npz", allow_pickle=True)["probs"]
top = test_classifier["p"].nlargest(4).index

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, i in zip(axes, top):
    image = cv2.cvtColor(cv2.imread(str(TEST_DIR / test_ids[i])), cv2.COLOR_BGR2RGB)
    mask = cv2.resize(test_probs[i], image.shape[1::-1], interpolation=cv2.INTER_LINEAR) > 0.7 * 255
    image[mask] = (0.6 * image[mask] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)
    ax.imshow(image)
    ax.set_title(f"{test_ids[i]}, p = {test_classifier['p'][i]:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.savefig(OUT_DIR / "test_examples.png", dpi=80)
plt.show()